# CSV Analyzer in Jupyter

This notebook does three things:

1. **Analyzes a CSV file** and answers questions like `average salary` or `max age`
2. **Downloads a CSV from a link** and accepts **CSV only** (JSON, HTML etc. are rejected)
3. **Runs the API and a client inside the notebook**, so you can see how they talk to each other

**How to use it:** run the cells from top to bottom with `Shift + Enter`.
Cells marked **(interactive)** ask you to type something. Type your question in the box that appears, and type `quit` to move on.

You need these packages: `pandas`, `requests`, `flask`, `matplotlib`.

In [ ]:
# Run this cell only if a package is missing (remove the # first)
# !pip install pandas requests flask matplotlib

In [ ]:
import json
import threading
from urllib.parse import urlparse

import pandas as pd
import requests

## Step 1: Create the example data

This cell writes `employees.csv` next to the notebook, then reads it back with pandas.

In [ ]:
csv_text = """name,department,age,salary,experience
Anna,Sales,30,50000,5
Bo,Sales,40,60000,15
Cy,IT,35,80000,10
Di,IT,45,90000,20
Ed,HR,28,45000,3
Fay,HR,38,55000,12
Gus,Finance,42,75000,18
Hana,Finance,31,65000,7
Ivan,IT,27,70000,3
Jo,Sales,33,52000,8
Kim,Marketing,29,48000,4
Lena,Marketing,36,58000,11
Marco,IT,50,95000,25
Nora,Finance,26,60000,2
Omar,Sales,45,68000,20
"""

with open("employees.csv", "w") as file:
    file.write(csv_text)

table = pd.read_csv("employees.csv")
table

## Step 2: The analyzer class

- `CSVError` is our own error type, used for any problem with the data or the question.
- `CSVAnalyzer` holds a table and has one method per job:

| Method | What it does |
|---|---|
| `info()` | Number of rows and the column names |
| `calculate(action, column)` | One calculation (average, sum, max, min, median) on a number column |
| `group_by(action, column, group)` | The same calculation for each group, e.g. per department |
| `answer(question)` | Reads a text question and calls the right method |

In [ ]:
class CSVError(Exception):
    """Our own error: used when the data or the question is not valid."""


class CSVAnalyzer:
    ACTIONS = {"average": "mean", "sum": "sum", "max": "max", "min": "min", "median": "median"}

    def __init__(self, table):
        self.table = table

    def info(self):
        return {"rows": len(self.table), "columns": list(self.table.columns)}

    def _check_number_column(self, column):
        if column not in self.table.columns:
            raise CSVError(f"Column '{column}' not found. Columns: {list(self.table.columns)}")
        if not pd.api.types.is_numeric_dtype(self.table[column]):
            raise CSVError(f"Column '{column}' is not a number column")

    def _check_action(self, action):
        if action not in self.ACTIONS:
            raise CSVError(f"Unknown action '{action}'. Use: {', '.join(self.ACTIONS)}")

    def calculate(self, action, column):
        self._check_action(action)
        self._check_number_column(column)
        result = getattr(self.table[column], self.ACTIONS[action])()
        return round(result.item(), 2)

    def group_by(self, action, column, group):
        self._check_action(action)
        self._check_number_column(column)
        if group not in self.table.columns:
            raise CSVError(f"Column '{group}' not found. Columns: {list(self.table.columns)}")
        result = getattr(self.table.groupby(group)[column], self.ACTIONS[action])()
        return result.round(2).to_dict()

    def answer(self, question):
        words = question.lower().split()

        if words == ["count"]:
            return {"answer": len(self.table)}
        if words == ["columns"]:
            return {"answer": list(self.table.columns)}

        if len(words) == 2:                                   # "average salary"
            action, column = words
            return {"action": action, "column": column,
                    "answer": self.calculate(action, column)}

        if len(words) == 4 and words[2] == "by":             # "average salary by department"
            action, column, _, group = words
            return {"action": action, "column": column, "group": group,
                    "answer": self.group_by(action, column, group)}

        raise CSVError("Question must look like: 'average salary' or 'average salary by department'")

## Step 3: Ask questions about the file

`ask()` sends a question to the analyzer and prints the answer or the error.

In [ ]:
analyzer = CSVAnalyzer(table)
print(analyzer.info())


def ask(question, analyzer=analyzer):
    try:
        result = analyzer.answer(question)
        print(f"{question}  ->  {result['answer']}")
    except CSVError as error:
        print(f"{question}  ->  Error: {error}")


for question in ["count", "columns", "average salary", "max age", "min experience",
                 "sum salary", "median salary", "average salary by department",
                 "average name", "average bonus"]:
    ask(question)

### Ask your own questions **(interactive)**

Examples: `average salary`, `max age`, `average salary by department`. Type `quit` to stop.

In [ ]:
while True:
    question = input("Your question (or quit): ").strip()
    if question.lower() == "quit":
        print("Goodbye!")
        break
    ask(question)

### A quick chart

Notebooks can draw charts, which the console version cannot.

In [ ]:
import matplotlib.pyplot as plt

table.groupby("department")["salary"].mean().sort_values().plot(
    kind="barh", title="Average salary by department")
plt.xlabel("Average salary")
plt.show()

## Step 4: Download a CSV from a link (CSV only)

`CSVFetcher.fetch(url)` follows these steps:

1. Check the link starts with `http://` or `https://`
2. Send the web request with `requests.get()`
3. Check the server answered with status 200
4. **Accept CSV only**: the link ends in `.csv` or the type contains "csv", and the type is not HTML, JSON or XML
5. Check the size (maximum 5 MB) and that pandas can read it as a table

In [ ]:
import io

MAX_SIZE = 5 * 1024 * 1024  # 5 MB


class CSVFetcher:
    def __init__(self, timeout=10):
        self.timeout = timeout

    def fetch(self, url):
        parsed = urlparse(url)
        if parsed.scheme not in ("http", "https") or not parsed.netloc:
            raise CSVError("Invalid URL. It must start with http:// or https://")

        try:
            response = requests.get(url, timeout=self.timeout)
        except requests.exceptions.RequestException as error:
            raise CSVError(f"Could not download the data: {error}")

        if response.status_code != 200:
            raise CSVError(f"The other server answered with status {response.status_code}")

        content_type = response.headers.get("Content-Type", "").lower()
        url_is_csv = parsed.path.lower().endswith(".csv")
        type_is_csv = "csv" in content_type
        type_is_other = "html" in content_type or "json" in content_type or "xml" in content_type

        if type_is_other or not (url_is_csv or type_is_csv):
            raise CSVError("Only CSV data is accepted (got: " + (content_type or "unknown") + ")")

        if len(response.content) > MAX_SIZE:
            raise CSVError("File is too big (maximum 5 MB)")

        try:
            remote_table = pd.read_csv(io.StringIO(response.text))
        except Exception:
            raise CSVError("The file could not be read as a CSV table")

        if remote_table.empty:
            raise CSVError("The CSV file has no data rows")

        return remote_table

### A small "other source" on your own computer

To try the fetcher without the internet, this cell starts a tiny web server **inside the notebook** (port 8000).
It shares the notebook folder, so `employees.csv` becomes available at `http://127.0.0.1:8000/employees.csv`.
It also creates `data.json` so we can test that non-CSV data is rejected.

In [ ]:
import functools
import http.server
import socketserver

with open("data.json", "w") as file:
    file.write('{"a": 1}')


class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args):   # hide the request log lines
        pass


socketserver.TCPServer.allow_reuse_address = True
try:
    file_server = socketserver.ThreadingTCPServer(
        ("127.0.0.1", 8000), functools.partial(QuietHandler, directory="."))
    threading.Thread(target=file_server.serve_forever, daemon=True).start()
    print("File server running: http://127.0.0.1:8000/employees.csv")
except OSError:
    print("Port 8000 is already in use. Run the cleanup cell at the end, or restart the kernel.")

In [ ]:
fetcher = CSVFetcher()

# 1) a real CSV: accepted
remote_table = fetcher.fetch("http://127.0.0.1:8000/employees.csv")
remote_analyzer = CSVAnalyzer(remote_table)
print(remote_analyzer.info())
ask("average salary", remote_analyzer)
ask("average salary by department", remote_analyzer)

# 2) JSON, a missing file and a bad link: rejected
for bad_url in ["http://127.0.0.1:8000/data.json",
                "http://127.0.0.1:8000/nofile.csv",
                "ftp://example.com/data.csv"]:
    try:
        fetcher.fetch(bad_url)
    except CSVError as error:
        print(f"{bad_url}  ->  Error: {error}")

## Step 5: Run the API inside the notebook

Now the same classes are wrapped in a Flask web API with two endpoints:

| Endpoint | What it does |
|---|---|
| `/analyze?url=<csv link>` | Rows and columns of the CSV |
| `/ask?url=<csv link>&question=<question>` | The answer to one question |

The API runs in a background thread on port 5000, so the notebook stays usable.

In [ ]:
import logging

from flask import Flask, jsonify, request
from werkzeug.serving import make_server

logging.getLogger("werkzeug").setLevel(logging.ERROR)   # hide the request log lines

app = Flask("csv_api")


@app.route("/analyze")
def analyze():
    url = request.args.get("url")
    if not url:
        return jsonify(error="Missing 'url' parameter"), 400
    try:
        return jsonify(CSVAnalyzer(fetcher.fetch(url)).info())
    except CSVError as error:
        return jsonify(error=str(error)), 400


@app.route("/ask")
def ask_route():
    url = request.args.get("url")
    question = request.args.get("question")
    if not url or not question:
        return jsonify(error="Missing 'url' or 'question' parameter"), 400
    try:
        return jsonify(CSVAnalyzer(fetcher.fetch(url)).answer(question))
    except CSVError as error:
        return jsonify(error=str(error)), 400


try:
    api_server = make_server("127.0.0.1", 5000, app)
    threading.Thread(target=api_server.serve_forever, daemon=True).start()
    print("API running: http://127.0.0.1:5000")
except OSError:
    print("Port 5000 is already in use. Run the cleanup cell at the end, or restart the kernel.")

## Step 6: Use the API from code (the client)

`api_ask()` sends the CSV link and the question to the API. Passing `params=` builds the link for us
and converts spaces to `%20` automatically.

In [ ]:
API_URL = "http://127.0.0.1:5000"
CSV_LINK = "http://127.0.0.1:8000/employees.csv"


def api_ask(csv_url, question):
    """Send one question to the API. Returns (ok, data)."""
    try:
        response = requests.get(f"{API_URL}/ask",
                                params={"url": csv_url, "question": question}, timeout=20)
    except requests.exceptions.ConnectionError:
        return False, {"error": "Cannot reach the API. Did you run the Step 5 cell?"}
    return response.status_code == 200, response.json()


for question in ["average salary", "max age", "count", "average salary by department", "average name"]:
    ok, data = api_ask(CSV_LINK, question)
    print(f"{question}  ->  {data['answer'] if ok else 'Error: ' + data['error']}")

# The API refuses anything that is not CSV
ok, data = api_ask("http://127.0.0.1:8000/data.json", "count")
print("data.json  ->", data)

### Ask the API your own questions **(interactive)**

Type a question and the notebook sends it to the API. Type `quit` to stop.

In [ ]:
csv_link = input("CSV link (press Enter to use the example): ").strip() or CSV_LINK

while True:
    question = input("Your question (or quit): ").strip()
    if question.lower() == "quit":
        print("Goodbye!")
        break
    ok, data = api_ask(csv_link, question)
    print("Answer:", data["answer"] if ok else "Error: " + data["error"])

You can also test the API in your browser while the notebook is running:

```
http://127.0.0.1:5000/analyze?url=http://127.0.0.1:8000/employees.csv
http://127.0.0.1:5000/ask?url=http://127.0.0.1:8000/employees.csv&question=average salary
```

## Cleanup

Run this cell when you are done. It stops both servers and frees ports 5000 and 8000.
(Restarting the kernel does the same.)

In [ ]:
for name in ["api_server", "file_server"]:
    server = globals().get(name)
    if server is not None:
        server.shutdown()
        server.server_close()
        print(name, "stopped")

## What next?

- Replace `employees.csv` with your own file (use lowercase column names for easy questions)
- Add a `count by department` question
- Show more charts with `matplotlib`